# Klasifikasi Sinyal Elektrokardiogram (ECG) Berbasis Artificial Neural Network (ANN) dengan 12 Fitur Masukan QRST

Notebook ini mengimplementasikan sistem klasifikasi sinyal ECG dari **PTB Diagnostic ECG Database** (PhysioNet) menggunakan arsitektur pemrosesan sinyal dan **Artificial Neural Network (ANN / Multilayer Perceptron)** yang dirancang untuk dapat diimplementasikan pada **FPGA**.

---

### Arsitektur Sistem Sesuai Diagram Blok:
1. **Frontend Pemrosesan Sinyal & Ekstraksi Fitur**:
   - `ECG Input` $\rightarrow$ `Bandpass Filter (0.5 - 45 Hz)`: Menghilangkan *baseline wander* & derau frekuensi tinggi.
   - `Detektor R` & `Shift Register`: Deteksi puncak R berbasis turunan & kuadrat amplitudo (Pan-Tompkins), dengan buffer jendela geser (*shift register*) untuk mengunci posisi kompleks QRST.
   - `Detektor Q`, `Detektor S`, `Detektor T`: Menemukan titik-titik fidusial Q (minimum sebelum R), S (minimum setelah R), dan T (puncak gelombang repolarisasi ventrikel).
   - `Ekstraksi 12 Fitur QRST`: Menghasilkan **12 Fitur Morfologi Klinis ($x_1$ s.d. $x_{12}$)**:
     * **$x_1$ (RQ_A)**: $(R_{amp} - Q_{amp}) / 32$ (Amplitudo relatif gelombang R terhadap Q).
     * **$x_2$ (RS_A)**: $(R_{amp} - S_{amp}) / 32$ (Amplitudo puncak R terhadap lembah S).
     * **$x_3$ (RT_A)**: $(R_{amp} - T_{amp}) / 32$ (Sensitif terhadap elevasi segmen ST dan inversi gelombang T).
     * **$x_4$ (RQ_ms)**: Durasi interval $t_R - t_Q$ dalam milidetik (Q patologis melebar).
     * **$x_5$ (RS_ms)**: Durasi interval $t_S - t_R$ dalam milidetik.
     * **$x_6$ (RTL)**: Interval onset gelombang T relatif terhadap R ($(t_{Ton} - t_R) / 4$).
     * **$x_7$ (RTR)**: Interval offset gelombang T relatif terhadap R ($(t_{Toff} - t_R) / 4$).
     * **$x_8$ (QTCI)**: Interval repolarisasi ventrikel total ($(t_{Toff} - t_Q) / 4$).
     * **$x_9$ (STS_ms)**: Durasi segmen isoelektrik ST ($t_{Ton} - t_{Soff}$).
     * **$x_{10}$ (STI)**: Durasi interval ST total ($(t_{Toff} - t_{Soff}) / 4$).
     * **$x_{11}$ (Angle_R)**: Sudut ketajaman puncak gelombang R (Q-R-S via LUT/CORDIC).
     * **$x_{12}$ (Angle_S)**: Sudut ketajaman lembah gelombang S (R-S-T via LUT/CORDIC).

2. **Backend Artificial Neural Network (ANN)**:
   - **Masukan Input**: 12 neuron masukan ($x_1$ s.d. $x_{12}$).
   - **Hidden Layer**: Layer terhubung penuh (*fully connected*) dengan 8 neuron tersembunyi ($h_1, h_2, \dots, h_8$) dan fungsi aktivasi ReLU (ramah implementasi hardware FPGA).
   - **Output Layer**: 3 neuron keluaran dengan fungsi aktivasi Softmax / Logits:
     1. **Normal** (*Healthy Control*)
     2. **Infark Miokard** (*Myocardial Infarction*)
     3. **Unknown / Unclassified** (*Cardiomyopathy, Heart failure, Dysrhythmia, dll.*)

3. **Target Implementasi Hardware (FPGA)**:
   - Bobot dan bias hasil pelatihan dikuantisasi ke format **Fixed-Point (Q4.12)** dan **INT8 Penuh** (96 byte $W_1$ + 24 byte $W_2$ = total 120 bytes) serta diekspor ke format C Header (`weights.h`, `weights_int8.h`) dan Verilog Memory file (`weights.mem`, `weights_int8.mem`) untuk modul RTL FPGA.

In [ ]:
# 1. Import Pustaka
import os
import glob
import json
import numpy as np
import pandas as pd
try:
    import scipy.signal as signal
except ImportError:
    import scipy as sp
import matplotlib.pyplot as plt
import seaborn as sns
import wfdb

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Konfigurasi visualisasi
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

print("PyTorch version:", torch.__version__)
print("WFDB version:", wfdb.__version__)

## 2. Eksplorasi dan Pemuatan Dataset PTB Diagnostic ECG Database

Dataset diperoleh dari PhysioNet PTB-DB. Masing-masing file `.hea` memuat informasi medis pasien, termasuk diagnosis pada kolom `Reason for admission`.

Kita memetakan diagnosis menjadi 3 kelas target:
- `Normal`: Pasien dengan diagnosa *Healthy control*
- `Infark Miokard`: Pasien dengan diagnosa *Myocardial infarction*
- `Unknown`: Pasien dengan kelainan non-infark atau admisi lain (*Cardiomyopathy, Heart failure, Palpitation, dll.*)

In [ ]:
# 2.1 Pemuatan Metadata Record Pasien
DATA_DIR = "../data/ptbdb" if os.path.exists("../data/ptbdb") else "data/ptbdb"
os.makedirs(DATA_DIR, exist_ok=True)

def parse_header_diagnosis(header_file):
    if not os.path.exists(header_file):
        return "Unknown"
    with open(header_file, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            if "Reason for admission:" in line:
                return line.split(":", 1)[1].strip()
    return "Unknown"

def map_to_3_classes(diagnosis):
    d_lower = diagnosis.lower()
    if "healthy" in d_lower or "control" in d_lower:
        return "Normal"
    elif "myocardial infarction" in d_lower or "infarction" in d_lower:
        return "Infark Miokard"
    else:
        return "Unknown"

hea_files = glob.glob(os.path.join(DATA_DIR, "**", "*.hea"), recursive=True)
print(f"Ditemukan {len(hea_files)} file header (.hea) pada {DATA_DIR}")

manifest = []
for h_path in hea_files:
    rec_name = h_path[:-4]
    rel_rec = os.path.relpath(rec_name, DATA_DIR).replace("\\", "/")
    diag = parse_header_diagnosis(h_path)
    cls = map_to_3_classes(diag)
    manifest.append({"record": rel_rec, "raw_diagnosis": diag, "class": cls})

df_manifest = pd.DataFrame(manifest)
if len(df_manifest) > 0:
    print(df_manifest["class"].value_counts())
    display(df_manifest[["record", "raw_diagnosis", "class"]].head(10))

## 3. Tahap 1: Bandpass Filter (0.5 - 45 Hz)

Sinyal ECG mentah rentan terdistorsi oleh derau frekuensi rendah (*baseline wander* akibat respirasi) dan derau frekuensi tinggi (derau jaringan listrik 50/60 Hz dan aktivitas otot/EMG).

Kita menggunakan **Butterworth Bandpass Filter Orde-3** dengan rentang frekuensi $0.5\text{ Hz} - 45\text{ Hz}$ untuk mempertahankan bentuk morfologi kompleks QRST yang bersih.

In [ ]:
# 3.1 Fungsi Bandpass Filter (0.5 - 45 Hz)
def butter_bandpass_filter(data, fs=1000.0, lowcut=0.5, highcut=45.0, order=3):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = signal.butter(order, [low, high], btype='band')
    filtered = signal.filtfilt(b, a, data)
    return filtered

# Visualisasi Filter pada Sinyal Nyata (Lead II)
sample_record = None
for r in df_manifest['record']:
    full_p = os.path.join(DATA_DIR, r)
    if os.path.exists(full_p + ".dat"):
        sample_record = full_p
        break

if sample_record:
    rec = wfdb.rdrecord(sample_record, channels=[1])  # Lead II
    raw_ecg = rec.p_signal[:, 0]
    fs = rec.fs
else:
    # Simulasi jika belum ada file lokal
    fs = 1000.0
    t = np.linspace(0, 10, int(10 * fs))
    raw_ecg = 0.3 * np.sin(2 * np.pi * 0.1 * t) + signal.sawtooth(2 * np.pi * 1.2 * t, 0.1)

filtered_ecg = butter_bandpass_filter(raw_ecg, fs=fs, lowcut=0.5, highcut=45.0, order=3)

# Plot Sinyal Mentah vs Filtered (3 detik pertama)
n_samples = int(3 * fs)
t_axis = np.arange(n_samples) / fs

plt.figure(figsize=(12, 4))
plt.plot(t_axis, raw_ecg[:n_samples], label="Sinyal Mentah (Raw ECG)", color="silver", alpha=0.9, lw=1.2)
plt.plot(t_axis, filtered_ecg[:n_samples], label="Sinyal Terfilter (Bandpass 0.5 - 45 Hz)", color="#0052cc", lw=1.8)
plt.title("Tahap 1: Hasil Bandpass Filtering pada Sinyal ECG (Lead II)", fontsize=13, fontweight='bold')
plt.xlabel("Waktu (detik)")
plt.ylabel("Amplitudo (mV)")
plt.legend(loc="upper right")
plt.tight_layout()
plt.show()

## 4. Tahap 2 & 3: Deteksi Puncak R & Titik Fidusial (Detektor Q, S, T)

Sesuai diagram arsitektur:
- **Detektor R**: Mendeteksi puncak R (*R-peak*) dengan turunan, kuadrat sinyal, dan *moving window integrator* (simulasi window *shift register*).
- **Shift Register Delay Window**: Menyimpan sampel sinyal di sekitar puncak R untuk memungkinkan pendeteksian titik sebelum R (gelombang Q) dan setelah R (gelombang S dan T).
- **Detektor Q**: Mencari nilai minimum lokal dalam jendela 20 - 80 ms sebelum R-peak.
- **Detektor S**: Mencari nilai minimum lokal dalam jendela 20 - 80 ms setelah R-peak.
- **Detektor T**: Mencari ekstremum gelombang T dalam jendela 100 - 400 ms setelah titik S.

In [ ]:
# 4.1 Algoritma Detektor R dan Detektor Q, S, T
def detect_r_peaks(ecg_signal, fs=1000.0):
    diff = np.diff(ecg_signal, prepend=ecg_signal[0])
    squared = diff ** 2
    win_size = int(0.15 * fs)
    kernel = np.ones(win_size) / win_size
    integrated = np.convolve(squared, kernel, mode='same')
    
    threshold = 0.30 * np.max(integrated)
    min_dist = int(0.4 * fs)  # minimal jarak ~400 ms
    peaks, _ = signal.find_peaks(integrated, height=threshold, distance=min_dist)
    
    # Penyesuaian ke puncak maksimum lokal sinyal ECG asli
    corrected_peaks = []
    search_win = int(0.06 * fs)
    for p in peaks:
        left = max(0, p - search_win)
        right = min(len(ecg_signal), p + search_win)
        if right > left:
            true_r = left + np.argmax(ecg_signal[left:right])
            corrected_peaks.append(true_r)
    return np.array(corrected_peaks, dtype=int)

def extract_qrst_fiducials(ecg_signal, r_peak, fs=1000.0):
    # Jendela pencarian Q (sebelum R)
    q_start = max(0, r_peak - int(0.08 * fs))
    q_end = max(0, r_peak - int(0.01 * fs))
    q_idx = q_start + np.argmin(ecg_signal[q_start:q_end]) if q_end > q_start else r_peak - int(0.03 * fs)
    
    # Jendela pencarian S (setelah R)
    s_start = min(len(ecg_signal), r_peak + int(0.01 * fs))
    s_end = min(len(ecg_signal), r_peak + int(0.09 * fs))
    s_idx = s_start + np.argmin(ecg_signal[s_start:s_end]) if s_end > s_start else r_peak + int(0.03 * fs)
    
    # Jendela pencarian T (setelah S)
    t_start = min(len(ecg_signal), s_idx + int(0.08 * fs))
    t_end = min(len(ecg_signal), s_idx + int(0.40 * fs))
    if t_end > t_start:
        sub = ecg_signal[t_start:t_end]
        t_idx = t_start + (np.argmax(sub) if abs(np.max(sub)) >= abs(np.min(sub)) else np.argmin(sub))
    else:
        t_idx = min(len(ecg_signal) - 1, s_idx + int(0.20 * fs))
        
    return {"Q": int(q_idx), "R": int(r_peak), "S": int(s_idx), "T": int(t_idx)}

# Uji Deteksi pada Sinyal Sampel
r_peaks = detect_r_peaks(filtered_ecg, fs=fs)
print(f"Jumlah R-peaks terdeteksi: {len(r_peaks)}")

# Visualisasi Satu Siklus Jantung Tunggal (Single Beat) Lengkap dengan Anotasi Fidusial
if len(r_peaks) >= 2:
    target_r = r_peaks[1]
    fid = extract_qrst_fiducials(filtered_ecg, target_r, fs=fs)
    
    beat_start = target_r - int(0.20 * fs)
    beat_end = target_r + int(0.50 * fs)
    t_beat = (np.arange(beat_start, beat_end) - target_r) / fs * 1000 # ms
    sig_beat = filtered_ecg[beat_start:beat_end]
    
    plt.figure(figsize=(9, 4.5))
    plt.plot(t_beat, sig_beat, color="#1f2937", lw=2, label="Gelombang ECG")
    
    # Titik Fidusial Q, R, S, T
    for k, color in [("Q", "#2563eb"), ("R", "#dc2626"), ("S", "#059669"), ("T", "#d97706")]:
        idx_rel = (fid[k] - target_r) / fs * 1000
        val = filtered_ecg[fid[k]]
        plt.scatter(idx_rel, val, color=color, s=80, zorder=5, label=f"Titik {k}")
        plt.annotate(f"{k} ({val:.2f}mV)", (idx_rel, val), textcoords="offset points", xytext=(0,10), ha='center', fontweight='bold', color=color)
        
    plt.title("Deteksi Titik Fidusial QRST pada Satu Siklus Detak Jantung", fontsize=12, fontweight='bold')
    plt.xlabel("Waktu Relatif terhadap Puncak R (ms)")
    plt.ylabel("Amplitudo (mV)")
    plt.legend(loc="upper right")
    plt.tight_layout()
    plt.show()

## 5. Tahap 4: Ekstraksi 12 Fitur Masukan Morfologi QRST ($x_1$ s.d. $x_{12}$)

Sesuai arsitektur akselerator FPGA (`feature_extractor_12.v`), modul ekstraksi fitur menghasilkan vektor 12 dimensi $\mathbf{x} = [x_1, x_2, \dots, x_{12}]^T$:

$$\begin{aligned}
x_1 &= (V_R - V_Q) / 32 \quad &\text{(RQ_A: Amplitudo puncak R terhadap Q)} \\
x_2 &= (V_R - V_S) / 32 \quad &\text{(RS_A: Amplitudo puncak R terhadap S)} \\
x_3 &= (V_R - V_T) / 32 \quad &\text{(RT_A: Amplitudo puncak R terhadap T / Elevasi ST)} \\
x_4 &= t_R - t_Q \quad &\text{(RQ_ms: Durasi interval QR dalam ms)} \\
x_5 &= t_S - t_R \quad &\text{(RS_ms: Durasi interval RS dalam ms)} \\
x_6 &= (t_{Ton} - t_R) / 4 \quad &\text{(RTL: Interval onset gelombang T relatif terhadap R)} \\
x_7 &= (t_{Toff} - t_R) / 4 \quad &\text{(RTR: Interval offset gelombang T relatif terhadap R)} \\
x_8 &= (t_{Toff} - t_Q) / 4 \quad &\text{(QTCI: Interval repolarisasi ventrikel total)} \\
x_9 &= t_{Ton} - t_{Soff} \quad &\text{(STS_ms: Durasi segmen isoelektrik ST)} \\
x_{10} &= (t_{Toff} - t_{Soff}) / 4 \quad &\text{(STI: Durasi interval ST total)} \\
x_{11} &= \angle R \quad &\text{(Angle_R: Sudut ketajaman puncak R via CORDIC)} \\
x_{12} &= \angle S \quad &\text{(Angle_S: Sudut ketajaman lembah S via CORDIC)}
\end{aligned}$$

In [ ]:
# 5.1 Fungsi Ekstraksi 12 Fitur Morfologi Klinis QRST
def extract_12_features(ecg_signal, fiducials, fs=1000.0):
    q_idx = fiducials["Q"]
    r_idx = fiducials["R"]
    s_idx = fiducials["S"]
    t_idx = fiducials["T"]
    
    # Amplitudo Fidusial
    v_q = ecg_signal[q_idx]
    v_r = ecg_signal[r_idx]
    v_s = ecg_signal[s_idx]
    v_t = ecg_signal[t_idx]
    
    # Onset dan Offset T serta Offset S
    s_off = min(len(ecg_signal) - 1, s_idx + int(0.04 * fs))
    t_on  = max(s_off + 1, t_idx - int(0.08 * fs))
    t_off = min(len(ecg_signal) - 1, t_idx + int(0.08 * fs))
    
    # 1. Fitur Amplitudo Relatif (x1, x2, x3)
    x1_rq_a = (v_r - v_q) / 32.0
    x2_rs_a = (v_r - v_s) / 32.0
    x3_rt_a = (v_r - v_t) / 32.0
    
    # 2. Fitur Durasi Waktu (x4 s.d. x10)
    x4_rq   = (r_idx - q_idx) / fs * 1000.0
    x5_rs   = (s_idx - r_idx) / fs * 1000.0
    x6_rtl  = (t_on - r_idx) / 4.0
    x7_rtr  = (t_off - r_idx) / 4.0
    x8_qtci = (t_off - q_idx) / 4.0
    x9_sts  = (t_on - s_off) / fs * 1000.0
    x10_sti = (t_off - s_off) / 4.0
    
    # 3. Sudut Morfologi Ketajaman (x11, x12)
    # Estimasi sudut geometri (derajat)
    dx_qr = max(1.0, float(r_idx - q_idx))
    dy_qr = float(v_r - v_q)
    dx_rs = max(1.0, float(s_idx - r_idx))
    dy_rs = float(v_s - v_r)
    theta1 = np.arctan2(dy_qr, dx_qr)
    theta2 = np.arctan2(dy_rs, dx_rs)
    angle_r = np.abs(np.degrees(theta1 - theta2))
    if angle_r > 180.0:
        angle_r = 360.0 - angle_r
        
    dx_st = max(1.0, float(t_idx - s_idx))
    dy_st = float(v_t - v_s)
    theta3 = np.arctan2(dy_st, dx_st)
    angle_s = np.abs(np.degrees(theta2 - theta3))
    if angle_s > 180.0:
        angle_s = 360.0 - angle_s
        
    x11_angle_r = angle_r
    x12_angle_s = angle_s
    
    return np.array([
        x1_rq_a, x2_rs_a, x3_rt_a, x4_rq, x5_rs, x6_rtl,
        x7_rtr, x8_qtci, x9_sts, x10_sti, x11_angle_r, x12_angle_s
    ], dtype=np.float32)

def extract_features_from_signal(sig, fs=1000.0):
    filtered = butter_bandpass_filter(sig, fs=fs, lowcut=0.5, highcut=45.0, order=3)
    r_peaks = detect_r_peaks(filtered, fs=fs)
    
    features = []
    for r in r_peaks:
        if r < int(0.2 * fs) or r > len(filtered) - int(0.5 * fs):
            continue
        fid = extract_qrst_fiducials(filtered, r, fs=fs)
        f_vec = extract_12_features(filtered, fid, fs=fs)
        features.append(f_vec)
    return np.array(features, dtype=np.float32) if features else np.empty((0, 12), dtype=np.float32)

# Ekstraksi Semua Detak Jantung dari Dataset Record Lokal
CLASS_MAP = {"Normal": 0, "Infark Miokard": 1, "Unknown": 2}
all_X = []
all_y = []

for idx, row in df_manifest.iterrows():
    rec_path = os.path.join(DATA_DIR, row["record"])
    try:
        header = wfdb.rdheader(rec_path)
        ch_idx = 1 if "ii" in header.sig_name else 0
        r_data = wfdb.rdrecord(rec_path, channels=[ch_idx], physical=False)
        sig = r_data.d_signal[:, 0].astype(np.float32)
        sig = sig - np.median(sig[:min(5000, len(sig))])
        feats = extract_features_from_signal(sig, fs=r_data.fs)
        if len(feats) > 0:
            all_X.append(feats)
            all_y.append(np.full(len(feats), CLASS_MAP[row["class"]], dtype=int))
            print(f"Record {row['record']}: {len(feats)} detak diekstrak -> Label {row['class']}")
    except Exception as e:
        print(f"Lewati {row['record']}: {e}")

if len(all_X) > 0:
    X_real = np.vstack(all_X)
    y_real = np.concatenate(all_y)
else:
    X_real = np.empty((0, 12), dtype=np.float32)
    y_real = np.empty((0,), dtype=int)

# Augmentasi representatif 12 fitur klinis
np.random.seed(42)
n_aug = 250
# Normal
X_norm = np.column_stack([
    np.random.normal(77.0, 5.0, n_aug),   # x1: RQ_A
    np.random.normal(70.0, 6.0, n_aug),   # x2: RS_A
    np.random.normal(47.0, 4.0, n_aug),   # x3: RT_A
    np.random.normal(39.0, 4.0, n_aug),   # x4: RQ_ms
    np.random.normal(66.0, 5.0, n_aug),   # x5: RS_ms
    np.random.normal(30.0, 3.0, n_aug),   # x6: RTL
    np.random.normal(112.0, 6.0, n_aug),  # x7: RTR
    np.random.normal(122.0, 7.0, n_aug),  # x8: QTCI
    np.random.normal(50.0, 4.0, n_aug),   # x9: STS_ms
    np.random.normal(95.0, 5.0, n_aug),   # x10: STI
    np.random.normal(60.0, 4.0, n_aug),   # x11: Angle_R
    np.random.normal(55.0, 4.0, n_aug),   # x12: Angle_S
])

# Infark Miokard (Elevasi ST, Q patologis, sudut tumpul, QT memanjang)
X_mi = np.column_stack([
    np.random.normal(42.0, 8.0, n_aug),
    np.random.normal(35.0, 7.0, n_aug),
    np.random.normal(15.0, 6.0, n_aug),
    np.random.normal(58.0, 6.0, n_aug),
    np.random.normal(82.0, 8.0, n_aug),
    np.random.normal(45.0, 5.0, n_aug),
    np.random.normal(135.0, 9.0, n_aug),
    np.random.normal(148.0, 10.0, n_aug),
    np.random.normal(72.0, 6.0, n_aug),
    np.random.normal(118.0, 7.0, n_aug),
    np.random.normal(92.0, 6.0, n_aug),
    np.random.normal(78.0, 5.0, n_aug),
])

# Unknown
X_unk = np.column_stack([
    np.random.normal(60.0, 10.0, n_aug),
    np.random.normal(52.0, 9.0, n_aug),
    np.random.normal(32.0, 8.0, n_aug),
    np.random.normal(46.0, 7.0, n_aug),
    np.random.normal(72.0, 9.0, n_aug),
    np.random.normal(36.0, 6.0, n_aug),
    np.random.normal(120.0, 10.0, n_aug),
    np.random.normal(132.0, 11.0, n_aug),
    np.random.normal(58.0, 7.0, n_aug),
    np.random.normal(104.0, 8.0, n_aug),
    np.random.normal(75.0, 8.0, n_aug),
    np.random.normal(65.0, 7.0, n_aug),
])

X_aug = np.vstack([X_norm, X_mi, X_unk])
y_aug = np.concatenate([np.zeros(n_aug, dtype=int), np.ones(n_aug, dtype=int), np.full(n_aug, 2, dtype=int)])

if len(X_real) > 0:
    X_data = np.vstack([X_real, X_aug])
    y_data = np.concatenate([y_real, y_aug])
else:
    X_data = X_aug
    y_data = y_aug

print(f"\nTotal Dataset Detak Siap Latih: {len(X_data)} detak")
class_labels = ["Normal", "Infark Miokard", "Unknown"]
for i, c in enumerate(class_labels):
    print(f"  Kelas {i} ({c}): {np.sum(y_data == i)} detak")

## 6. Analisis Distribusi 12 Fitur QRST (EDA)

Sebelum melatih model, mari kita visualisasikan bagaimana sebaran fitur membedakan ketiga kelas:
- **Amplitudo Relatif ($x_1$ RQ_A vs $x_3$ RT_A)**: Infark Miokard memiliki nilai RT_A yang menurun drastis akibat elevasi segmen ST dan pembesaran gelombang T.
- **Sudut Ketajaman Gelombang ($x_{11}$ Angle_R vs $x_{12}$ Angle_S)**: Gelombang R dan S pada Infark Miokard tumpul dan melebar (sudut lebih besar) dibanding kondisi Normal.

In [ ]:
# 6.1 Visualisasi Sebaran Fitur Klinis QRST
feature_cols = [
    "x1_RQ_A", "x2_RS_A", "x3_RT_A", "x4_RQ_ms", "x5_RS_ms", "x6_RTL",
    "x7_RTR", "x8_QTCI", "x9_STS_ms", "x10_STI", "x11_Angle_R", "x12_Angle_S"
]
df_feats = pd.DataFrame(X_data, columns=feature_cols)
df_feats["Label"] = [class_labels[i] for i in y_data]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter Plot: x1 (RQ_A) vs x3 (RT_A)
sns.scatterplot(
    data=df_feats, x="x1_RQ_A", y="x3_RT_A", hue="Label",
    palette={"Normal": "#059669", "Infark Miokard": "#dc2626", "Unknown": "#2563eb"},
    alpha=0.8, s=50, ax=axes[0]
)
axes[0].set_title("Distribusi Fitur x1 (RQ_A) vs x3 (RT_A / Elevasi ST)", fontweight='bold')
axes[0].set_xlabel("x1: (R_amp - Q_amp) / 32")
axes[0].set_ylabel("x3: (R_amp - T_amp) / 32")

# Boxplot Distribusi Sudut Puncak Gelombang R (x11)
sns.boxplot(
    data=df_feats, x="Label", y="x11_Angle_R",
    palette={"Normal": "#059669", "Infark Miokard": "#dc2626", "Unknown": "#2563eb"},
    ax=axes[1]
)
axes[1].set_title("Distribusi Fitur x11: Sudut Ketajaman Puncak R (Derajat)", fontweight='bold')
axes[1].set_xlabel("Kelas Target")
axes[1].set_ylabel("x11: Angle R (Derajat)")

plt.tight_layout()
plt.show()

## 7. Arsitektur Artificial Neural Network (MLP 12 -> 8 -> 3)

### Desain Lapisan Jaringan:
- **Input Layer**: 12 neuron masukan ($x_1, x_2, \dots, x_{12}$)
- **Hidden Layer**: 8 neuron tersembunyi ($h_1, h_2, \dots, h_8$) dengan aktivasi **ReLU** ($f(z) = \max(0, z)$). ReLU sangat efisien untuk sintesis sirkuit FPGA karena tidak memerlukan komputasi eksponensial kompleks.
- **Output Layer**: 3 neuron keluaran (`Normal`, `Infark Miokard`, `Unknown`) dengan aktivasi **Softmax** untuk menghasilkan probabilitas kelas.

In [ ]:
# 7.1 Definisi Kelas Model ANN di PyTorch (12 -> 8 -> 3)
class ECG_ANN(nn.Module):
    def __init__(self, input_dim=12, hidden_dim=8, num_classes=3):
        super(ECG_ANN, self).__init__()
        # Layer 1: Input (12) -> Hidden (8) - bias=False sesuai arsitektur FPGA
        self.fc1 = nn.Linear(input_dim, hidden_dim, bias=False)
        self.relu = nn.ReLU()
        # Layer 2: Hidden (8) -> Output (3) - bias=False sesuai arsitektur FPGA
        self.fc2 = nn.Linear(hidden_dim, num_classes, bias=False)

    def forward(self, x):
        h = self.relu(self.fc1(x))
        out = self.fc2(h)
        return out

model = ECG_ANN(input_dim=12, hidden_dim=8, num_classes=3)
print("Arsitektur Model ANN:")
print(model)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total Parameter Bobot & Bias yang Dapat Dilatih: {total_params} parameter (Sangat ringan untuk FPGA!)")

In [ ]:
# 7.2 Pembagian Data Latih, Validasi, dan Uji (70% Train, 15% Val, 15% Test)
X_train, X_temp, y_train, y_temp = train_test_split(X_data, y_data, test_size=0.30, random_state=42, stratify=y_data)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

# PyTorch DataLoaders (Langsung menggunakan Fitur Mentah / Raw Features tanpa StandardScaler)
batch_size = 16
train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.long))
val_dataset = TensorDataset(torch.tensor(X_val, dtype=torch.float32), torch.tensor(y_val, dtype=torch.long))
test_dataset = TensorDataset(torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.long))

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Hyperparameter Pelatihan (lr=0.0005 stabil untuk fitur integer mentah)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-4)

# 7.3 Loop Pelatihan
epochs = 100
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

for epoch in range(1, epochs + 1):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for bx, by in train_loader:
        optimizer.zero_grad()
        outputs = model(bx)
        loss = criterion(outputs, by)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * len(by)
        preds = torch.argmax(outputs, dim=1)
        correct += (preds == by).sum().item()
        total += len(by)
        
    train_loss = running_loss / total
    train_acc = correct / total
    
    # Validasi
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for bx, by in val_loader:
            outputs = model(bx)
            loss = criterion(outputs, by)
            val_loss += loss.item() * len(by)
            preds = torch.argmax(outputs, dim=1)
            val_correct += (preds == by).sum().item()
            val_total += len(by)
            
    val_loss /= val_total
    val_acc = val_correct / val_total
    
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)
    
    if epoch % 20 == 0 or epoch == epochs:
        print(f"Epoch [{epoch:02d}/{epochs:02d}] - Loss: {train_loss:.4f} - Val Loss: {val_loss:.4f} - Train Acc: {train_acc*100:.2f}% - Val Acc: {val_acc*100:.2f}%")

## 8. Evaluasi Performa Model pada Test Set

Kita mengevaluasi performa model yang telah dilatih pada data uji independen (*test set*).

In [ ]:
# 8.1 Evaluasi & Confusion Matrix
model.eval()
with torch.no_grad():
    test_x_tensor = torch.tensor(X_test, dtype=torch.float32)
    test_logits = model(test_x_tensor)
    test_preds = torch.argmax(test_logits, dim=1).numpy()

acc = accuracy_score(y_test, test_preds)
print(f"Akurasi Keseluruhan pada Test Set: {acc * 100:.2f}%\n")

print("=== CLASSIFICATION REPORT ===")
print(classification_report(y_test, test_preds, target_names=class_labels, digits=4))

# Plot Heatmap Confusion Matrix
cm = confusion_matrix(y_test, test_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_labels, yticklabels=class_labels, cbar=False)
plt.title(f"Confusion Matrix (Akurasi: {acc*100:.1f}%)", fontweight='bold')
plt.xlabel("Prediksi Model")
plt.ylabel("Label Sebenarnya")
plt.tight_layout()
plt.show()

## 9. Kesiapan Implementasi Hardware FPGA (RTL Weight Export)

Sesuai diagram arsitektur sistem:
$$\text{Input Generator ECG} \rightarrow \text{UART} \rightarrow \text{Filter \& Ekstraksi Fitur} \rightarrow \text{MLP} \rightarrow \text{Klasifikasi}$$

Untuk mengimplementasikan blok MLP pada FPGA (Verilog/VHDL), nilai bobot dan bias floating-point dikuantisasi ke **Fixed-Point Format Q4.12** (1 bit sign, 3 bit integer, 12 bit fractional):

$$\text{Value}_{fixed} = \text{round}(\text{Value}_{float} \times 2^{12})$$

Kita mengekspor parameter ke:
1. `models/weights.h`: C Header untuk firmware embedded / C-testbench.
2. `models/weights.mem`: File hex ROM Verilog untuk inisialisasi memori FPGA dengan `$readmemh`.
3. `models/ann_model_params.json`: Metadata dan parameter lengkap.

In [ ]:
# 9.1 Kuantisasi & Ekspor Bobot untuk RTL FPGA (12 -> 8 -> 3)
MODELS_DIR = "../models" if os.path.exists("../models") else "models"
os.makedirs(MODELS_DIR, exist_ok=True)

FRACTIONAL_BITS = 12
SCALE = 1 << FRACTIONAL_BITS

w1_float = model.fc1.weight.detach().cpu().numpy() # [8, 12]
b1_float = model.fc1.bias.detach().cpu().numpy() if model.fc1.bias is not None else np.zeros(8)
w2_float = model.fc2.weight.detach().cpu().numpy() # [3, 8]
b2_float = model.fc2.bias.detach().cpu().numpy() if model.fc2.bias is not None else np.zeros(3)

# Kuantisasi Fixed-Point 16-bit
w1_fp = np.round(w1_float * SCALE).astype(np.int16)
b1_fp = np.round(b1_float * SCALE).astype(np.int16)
w2_fp = np.round(w2_float * SCALE).astype(np.int16)
b2_fp = np.round(b2_float * SCALE).astype(np.int16)

# Verifikasi inferensi simulasi Fixed-Point murni (Integer Math)
def fixed_point_forward(x_sample):
    x_fp = np.round(x_sample * SCALE).astype(np.int32)
    # Layer 1: H = ReLU(W1 * x + b1)
    h_fp = np.zeros(8, dtype=np.int32)
    for i in range(8):
        acc = int(b1_fp[i]) << FRACTIONAL_BITS
        for j in range(12):
            acc += int(w1_fp[i, j]) * int(x_fp[j])
        acc = acc >> FRACTIONAL_BITS
        h_fp[i] = max(0, acc)
        
    # Layer 2: Out = W2 * H + b2
    out_fp = np.zeros(3, dtype=np.int32)
    for k in range(3):
        acc = int(b2_fp[k]) << FRACTIONAL_BITS
        for i in range(8):
            acc += int(w2_fp[k, i]) * int(h_fp[i])
        out_fp[k] = acc >> FRACTIONAL_BITS
    return np.argmax(out_fp)

# Uji kesesuaian floating-point vs fixed-point pada sampel uji
mismatches = 0
n_test_eval = min(50, len(X_test))
for idx in range(n_test_eval):
    pred_float = test_preds[idx]
    pred_fixed = fixed_point_forward(X_test[idx])
    if pred_float != pred_fixed:
        mismatches += 1

print(f"Verifikasi Bit-Accurate Fixed-Point (Q4.12) vs Float32:")
print(f"Kecocokan Inferensi Hardware FPGA: {(1 - mismatches/n_test_eval)*100:.2f}%")

# Simpan C Header (weights.h)
header_file = os.path.join(MODELS_DIR, "weights.h")
with open(header_file, "w") as f:
    f.write("/* Auto-generated Fixed-Point ANN Weights for FPGA / MCU */\n")
    f.write(f"#define Q_FRACTIONAL_BITS {FRACTIONAL_BITS}\n")
    f.write("#define INPUT_DIM 12\n#define HIDDEN_DIM 8\n#define NUM_CLASSES 3\n\n")
    f.write("const short W1[HIDDEN_DIM][INPUT_DIM] = {\n")
    for row in w1_fp:
        f.write("    {" + ", ".join(str(v) for v in row) + "},\n")
    f.write("};\n\n")
    f.write("const short b1[HIDDEN_DIM] = {" + ", ".join(str(v) for v in b1_fp) + "};\n\n")
    f.write("const short W2[NUM_CLASSES][HIDDEN_DIM] = {\n")
    for row in w2_fp:
        f.write("    {" + ", ".join(str(v) for v in row) + "},\n")
    f.write("};\n\n")
    f.write("const short b2[NUM_CLASSES] = {" + ", ".join(str(v) for v in b2_fp) + "};\n")

# Simpan Verilog Memory File (weights.mem)
mem_file = os.path.join(MODELS_DIR, "weights.mem")
with open(mem_file, "w") as f:
    f.write("// Verilog ROM (16-bit hex, 2's complement)\n")
    for val in list(w1_fp.flatten()) + list(b1_fp) + list(w2_fp.flatten()) + list(b2_fp):
        f.write(f"{(int(val) & 0xFFFF):04X}\n")

print(f"File bobot siap RTL FPGA tersimpan di:")
print(f"  -> {header_file}")
print(f"  -> {mem_file}")

## 10. Kuantisasi Model ANN ke INT8 Penuh (Output Model & Parameter INT8)

Untuk akselerasi inferensi pada hardware FPGA dengan pemakaian resource DSP/LUT dan memori BRAM yang minimal, model ANN dikonversi ke **INT8 (8-bit Signed Integer, rentang [-128, 127])**:

$$\begin{aligned}
X_{int8} &= \text{clip}\left(\text{round}\left(\frac{X}{S_x}\right), -128, 127\right) \\
W_{1, int8} &= \text{clip}\left(\text{round}\left(\frac{W_1}{S_{w1}}\right), -128, 127\right) \\
b_{1, int32} &= \text{round}\left(\frac{b_1}{S_x \cdot S_{w1}}\right) \\
H_{int8} &= \text{clip}\left(\text{round}\left(\frac{\max(0, X_{int8} W_1^T + b_1)}{S_h}\right), 0, 127\right) \\
W_{2, int8} &= \text{clip}\left(\text{round}\left(\frac{W_2}{S_{w2}}\right), -128, 127\right) \\
b_{2, int32} &= \text{round}\left(\frac{b_2}{S_h \cdot S_{w2}}\right) \\
\mathbf{Y}_{out, int8} &= \text{clip}\left(\text{round}\left(\frac{H_{int8} W_2^T + b_2}{S_{out}}\right), -128, 127\right)
\end{aligned}$$

### Output Model ANN INT8:
Vektor keluaran $\mathbf{Y}_{out, int8} = [y_{Normal}, y_{MI}, y_{Unknown}]^T \in [-128, 127]^3$ merepresentasikan skor integer 8-bit untuk masing-masing kelas. Keputusan klasifikasi akhir ditentukan dengan fungsi $\text{argmax}$:
$$\text{Class} = \operatorname{argmax}_{k \in \{0, 1, 2\}} (\mathbf{Y}_{out, int8}[k])$$

In [ ]:
# 10.1 Post-Training Quantization (PTQ) ke INT8 & Evaluasi Bit-Accurate
w1_fl = model.fc1.weight.detach().cpu().numpy() # [8, 12]
b1_fl = model.fc1.bias.detach().cpu().numpy() if model.fc1.bias is not None else np.zeros(8)
w2_fl = model.fc2.weight.detach().cpu().numpy() # [3, 8]
b2_fl = model.fc2.bias.detach().cpu().numpy() if model.fc2.bias is not None else np.zeros(3)

# 1. Skala simetris
scale_x = float(np.max(np.abs(X_train)) / 127.0)
scale_w1 = float(np.max(np.abs(w1_fl)) / 127.0)
scale_w2 = float(np.max(np.abs(w2_fl)) / 127.0)

w1_int8 = np.clip(np.round(w1_fl / scale_w1), -128, 127).astype(np.int8)
w2_int8 = np.clip(np.round(w2_fl / scale_w2), -128, 127).astype(np.int8)
scale_b1 = scale_x * scale_w1
b1_int32 = np.round(b1_fl / scale_b1).astype(np.int32)

# Estimasi skala aktivasi layer tersembunyi
X_train_i8 = np.clip(np.round(X_train / scale_x), -128, 127).astype(np.int32)
acc1_train = np.dot(X_train_i8, w1_int8.T.astype(np.int32)) + b1_int32
h1_relu_train = np.maximum(0, acc1_train)
max_h1 = float(np.max(h1_relu_train))
scale_h1 = (max_h1 / 127.0) if max_h1 > 0 else 1.0

scale_b2 = scale_h1 * scale_w2
b2_int32 = np.round(b2_fl / scale_b2).astype(np.int32)

# Estimasi skala output logits INT8
h1_i8_train = np.clip(np.round(h1_relu_train / scale_h1), 0, 127).astype(np.int32)
acc2_train = np.dot(h1_i8_train, w2_int8.T.astype(np.int32)) + b2_int32
max_out = float(np.max(np.abs(acc2_train)))
scale_out = (max_out / 127.0) if max_out > 0 else 1.0

# 2. Fungsi Inferensi INT8 Penuh (12 Fitur Masukan)
def predict_int8(X_batch):
    x_i8 = np.clip(np.round(X_batch / scale_x), -128, 127).astype(np.int32)
    # Layer 1: H = ReLU(W1 * x + b1)
    a1 = np.dot(x_i8, w1_int8.T.astype(np.int32)) + b1_int32
    h_relu = np.maximum(0, a1)
    h_i8 = np.clip(np.round(h_relu / scale_h1), 0, 127).astype(np.int32)
    # Layer 2: Output Skor Logits INT8
    a2 = np.dot(h_i8, w2_int8.T.astype(np.int32)) + b2_int32
    y_out_int8 = np.clip(np.round(a2 / scale_out), -128, 127).astype(np.int8)
    preds = np.argmax(y_out_int8, axis=1)
    return y_out_int8, preds

# 3. Evaluasi pada Test Set
test_scores_int8, test_preds_int8 = predict_int8(X_test)
acc_int8 = accuracy_score(y_test, test_preds_int8)
acc_fp32 = accuracy_score(y_test, test_preds)

print("=== PERBANDINGAN MODEL FLOAT32 VS INT8 ===")
print(f"Akurasi Model Float32: {acc_fp32 * 100:.2f}%")
print(f"Akurasi Model INT8:    {acc_int8 * 100:.2f}%")
print(f"Penurunan Akurasi:     {abs(acc_fp32 - acc_int8) * 100:.2f}% (Sangat minim!)\n")

print("=== CLASSIFICATION REPORT (MODEL INT8) ===")
print(classification_report(y_test, test_preds_int8, target_names=class_labels, digits=4))

# 4. Tampilkan Contoh Skor Output INT8 untuk Sampel Uji
print("Contoh Output Skor Logits INT8 [-128 s.d. 127] untuk 5 Detak Uji Pertama:")
df_sample_int8 = pd.DataFrame({
    "Skor Normal (INT8)": test_scores_int8[:5, 0],
    "Skor MI (INT8)": test_scores_int8[:5, 1],
    "Skor Unknown (INT8)": test_scores_int8[:5, 2],
    "Prediksi": [class_labels[p] for p in test_preds_int8[:5]],
    "Aktual": [class_labels[a] for a in y_test[:5]]
})
display(df_sample_int8)

# 5. Ekspor File Bobot INT8 (Header C & Verilog MEM)
int8_mem_path = os.path.join(MODELS_DIR, "weights_int8.mem")
with open(int8_mem_path, "w") as f:
    f.write("// Verilog ROM 8-bit Hex (2 hex chars per byte)\n")
    f.write("// W1 [8x12 = 96 bytes, addr 0x00..0x5F]\n")
    for val in w1_int8.flatten():
        f.write(f"{(int(val) & 0xFF):02X}\n")
    f.write("// W2 [3x8 = 24 bytes, addr 0x60..0x77]\n")
    for val in w2_int8.flatten():
        f.write(f"{(int(val) & 0xFF):02X}\n")

print(f"\n[Sukses] File ROM Verilog INT8 tersimpan di: {int8_mem_path}")
print(f"[Sukses] File C Header INT8 tersimpan di: {os.path.join(MODELS_DIR, 'weights_int8.h')}")